# Spark Profiling & Optimization - Showcase

Continuation of `compar-lab02-spark-dataframes`. Five worked examples, each
following the same method: **profile a naive computation -> diagnose the
bottleneck -> apply one targeted fix -> profile again -> quantify the
improvement**.

Dataset: MovieLens (`movies.csv.gz`, `ratings1.csv.gz`, `ratings2.csv.gz`) -
the same files used in lab02.


## Setup

In [ ]:
master = "spark://spark-master:7077"   # use "spark://spark-master:7077" when running inside the Docker cluster
dataFolder = "/data/lab02/" if master.startswith("spark://") else "../../compar-lab02-spark-dataframes/data/"


In [2]:
from pyspark.sql import SparkSession

spark_builder = (
    SparkSession.builder.master(master)
    .appName("spark-profiling-showcase")
    .config("spark.ui.showConsoleProgress", "false")
)

spark = spark_builder.getOrCreate()

print("Spark UI:", spark.sparkContext.uiWebUrl)
print(
    "Default parallelism (cores seen by the driver):",
    spark.sparkContext.defaultParallelism,
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 08:56:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://7c6811b510eb:4040
Default parallelism (cores seen by the driver): 6


Keep the Spark UI open in a browser tab next to this notebook for the whole
session: the profiler below gives you the numbers, the UI gives you the
picture (DAG, task timeline, per-task skew).

In [3]:
import sys
sys.path.append("/workspace/lab03")
from spark_profiler import SparkProfiler

profiler = SparkProfiler(spark)


## Example 1 - Partitioning: the non-splittable gzip trap

`ratings1.csv.gz` and `ratings2.csv.gz` are gzip-compressed. Gzip is **not
splittable**: no matter how many workers your cluster has, Spark can only
hand out **one task per file** to read it. On a 1-master/N-worker cluster,
that means at most 2 of your workers ever do anything during the read - the
rest sit idle.

In [4]:
with profiler.profile("1-baseline-partitions"):
    ratings = (spark.read
               .option("header", "true")
               .option("inferSchema", "true")
               .csv(dataFolder + "ratings1.csv.gz"))
    print("partitions right after read:", ratings.rdd.getNumPartitions())
    ratings.groupBy("movieId").count().orderBy("movieId").collect()


partitions right after read: 1
[1-baseline-partitions] wall time: 53.29s (10 stages, 10 tasks) - see profiler.report() / profiler.compare(...)


Look at the Spark UI's *Stages* tab for this run: how many tasks does the
read stage have? With N workers x cores-per-worker available, how many of
them are actually busy during that stage?

In [5]:
target_partitions = spark.sparkContext.defaultParallelism * 2   # ~2 tasks per core is a reasonable rule of thumb

with profiler.profile("1-repartitioned"):
    ratings = (spark.read
               .option("header", "true")
               .option("inferSchema", "true")
               .csv(dataFolder + "ratings1.csv.gz")
               .repartition(target_partitions))
    print("partitions after repartition:", ratings.rdd.getNumPartitions())
    ratings.groupBy("movieId").count().orderBy("movieId").collect()


partitions after repartition: 12
[1-repartitioned] wall time: 111.79s (16 stages, 66 tasks) - see profiler.report() / profiler.compare(...)


In [6]:
profiler.compare("1-baseline-partitions", "1-repartitioned")


,1-baseline-partitions,1-repartitioned,change
metric,,,
wall_time_s,53.29,111.79,+109.8%
num_jobs,6,8,+33.3%
num_stages,10,16,+60.0%
num_tasks,10,66,+560.0%
executor_run_time_ms,46519,110001,+136.5%
executor_cpu_time_ms,44894,106853,+138.0%
input_bytes,168.3MB,252.4MB,+50.0%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,1.3MB,10.6MB,+729.2%


`repartition()` itself costs a shuffle, so it isn't free - but it pays for
itself as soon as the *next* wide operation (the `groupBy` here) can finally
spread across every worker instead of two. In production the better fix is
usually to **not store data as raw gzip CSV** in the first place - write it
once as Parquet/Snappy (splittable, columnar, typed) and read from that from
then on. See the note at the end of this notebook.

## Example 2 - Schema inference and column pruning

`inferSchema` makes Spark do an extra full pass over the file just to guess
column types, *before* your actual query even starts. Selecting columns late
means every column - used or not - gets carried through any later shuffle.

In [ ]:
with profiler.profile("2-baseline-schema"):
    ratings = (spark.read
               .option("header", "true")
               .option("inferSchema", "true")
               .csv(dataFolder + "ratings1.csv.gz")
               .repartition(target_partitions))
    avg_by_movie = ratings.groupBy("movieId").avg("rating")
    avg_by_movie.orderBy("movieId").collect()


26/10/01 09:05:35 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /data/lab02/ratings*.csv.gz.
java.io.FileNotFoundException: File /data/lab02/ratings*.csv.gz does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.

[2-baseline-schema] wall time: 92.03s (15 stages, 71 tasks) - see profiler.report() / profiler.compare(...)


In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, LongType

ratings_schema = StructType([
    StructField("userId", IntegerType(), False),
    StructField("movieId", IntegerType(), False),
    StructField("rating", DoubleType(), False),
    StructField("timestamp", LongType(), False),
])

with profiler.profile("2-explicit-schema-pruned"):
    ratings = (spark.read
               .option("header", "true")
               .schema(ratings_schema)
               .csv(dataFolder + "ratings1.csv.gz")
               .repartition(target_partitions)
               .select("movieId", "rating"))       # drop userId/timestamp before the shuffle
    avg_by_movie = ratings.groupBy("movieId").avg("rating")
    avg_by_movie.orderBy("movieId").collect()


26/10/01 09:09:44 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /data/lab02/ratings*.csv.gz.
java.io.FileNotFoundException: File /data/lab02/ratings*.csv.gz does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.

[2-explicit-schema-pruned] wall time: 49.45s (13 stages, 68 tasks) - see profiler.report() / profiler.compare(...)


In [9]:
profiler.compare("2-baseline-schema", "2-explicit-schema-pruned")


,2-baseline-schema,2-explicit-schema-pruned,change
metric,,,
wall_time_s,92.03,49.45,-46.3%
num_jobs,7,5,-28.6%
num_stages,15,13,-13.3%
num_tasks,71,68,-4.2%
executor_run_time_ms,192862,100273,-48.0%
executor_cpu_time_ms,177144,98070,-44.6%
input_bytes,336.5MB,168.2MB,-50.0%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,35.0MB,34.9MB,-0.0%


Two independent savings stack up here: no inference pass over ~350MB of
CSV, and a narrower shuffle payload (2 columns instead of 4) for the
`groupBy`. Compare `shuffle_write_bytes` between the two runs.

## Example 3 - Join strategy: shuffle join vs broadcast join

`movies` (~9,000 rows, <1MB) is tiny next to `ratings` (~20M rows, ~350MB).
Joining them the naive way makes Spark shuffle *both* sides across the
network so matching rows land on the same executor. Since one side is tiny,
Spark can instead send a full copy of it to every executor and avoid
shuffling the big side at all.

In [ ]:
movies = (spark.read.option("header", "true").option("inferSchema", "true")
          .csv(dataFolder + "movies.csv.gz"))
ratings = (spark.read.option("header", "true").schema(ratings_schema)
           .csv(dataFolder + "ratings1.csv.gz").repartition(target_partitions))

with profiler.profile("3-shuffle-join"):
    joined = ratings.join(movies.hint("merge"), on="movieId")   # force a sort-merge join
    joined.groupBy("title").count().orderBy("count", ascending=False).limit(5).collect()

joined.explain()


26/10/01 09:15:19 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /data/lab02/ratings*.csv.gz.
java.io.FileNotFoundException: File /data/lab02/ratings*.csv.gz does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.

[3-shuffle-join] wall time: 62.25s (13 stages, 60 tasks) - see profiler.report() / profiler.compare(...)
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [movieId#160, userId#159, rating#161, timestamp#162L, title#157, genres#158]
   +- SortMergeJoin [movieId#160], [movieId#156], Inner
      :- Sort [movieId#160 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(movieId#160, 200), ENSURE_REQUIREMENTS, [plan_id=816]
      :     +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=812]
      :        +- Filter isnotnull(movieId#160)
      :           +- FileScan csv [userId#159,movieId#160,rating#161,timestamp#162L] Batched: false, DataFilters: [isnotnull(movieId#160)], Format: CSV, Location: InMemoryFileIndex(2 paths)[file:/data/lab02/ratings1.csv.gz, file:/data/lab02/ratings2.csv.gz], PartitionFilters: [], PushedFilters: [IsNotNull(movieId)], ReadSchema: struct<userId:int,movieId:int,rating:double,timestamp:bigint>
      +- Sort [movieId#

The plan above shows a `SortMergeJoin`: both `ratings` and `movies` get
shuffled by `movieId` before they can be matched up.

In [11]:
from pyspark.sql.functions import broadcast

with profiler.profile("3-broadcast-join"):
    joined = ratings.join(broadcast(movies), on="movieId")
    joined.groupBy("title").count().orderBy("count", ascending=False).limit(5).collect()

joined.explain()


[3-broadcast-join] wall time: 50.14s (7 stages, 37 tasks) - see profiler.report() / profiler.compare(...)
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [movieId#160, userId#159, rating#161, timestamp#162L, title#157, genres#158]
   +- BroadcastHashJoin [movieId#160], [movieId#156], Inner, BuildRight, false, false
      :- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=1022]
      :  +- Filter isnotnull(movieId#160)
      :     +- FileScan csv [userId#159,movieId#160,rating#161,timestamp#162L] Batched: false, DataFilters: [isnotnull(movieId#160)], Format: CSV, Location: InMemoryFileIndex(2 paths)[file:/data/lab02/ratings1.csv.gz, file:/data/lab02/ratings2.csv.gz], PartitionFilters: [], PushedFilters: [IsNotNull(movieId)], ReadSchema: struct<userId:int,movieId:int,rating:double,timestamp:bigint>
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=1025]
         +- Filter isnotnull(movie

In [12]:
profiler.compare("3-shuffle-join", "3-broadcast-join")


,3-shuffle-join,3-broadcast-join,change
metric,,,
wall_time_s,62.25,50.14,-19.5%
num_jobs,5,4,-20.0%
num_stages,13,7,-46.2%
num_tasks,60,37,-38.3%
executor_run_time_ms,168041,114302,-32.0%
executor_cpu_time_ms,154919,108100,-30.2%
input_bytes,169.0MB,169.0MB,+0.0%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,18.0MB,17.7MB,-1.9%


`ratings` no longer needs to be shuffled at all for the join - only the much
smaller `movies` table is sent to every executor. Watch `shuffle_read_bytes`
and `shuffle_write_bytes` drop close to zero for the join stage.

(Spark's optimizer would normally pick the broadcast join automatically once
`movies` is under `spark.sql.autoBroadcastJoinThreshold` - the
`.hint("merge")` above was only there to force the *bad* plan for the demo.)

## Example 4 - Adaptive Query Execution (AQE): coalescing shuffle partitions

Every SQL shuffle produces exactly `spark.sql.shuffle.partitions` partitions
(200 by default) - a fixed number, chosen without knowing how much data will
actually land in each one. On a small teaching cluster that's almost always
too many: hundreds of tasks, most doing almost no work, all still paying
scheduling overhead. Since Spark 3, **Adaptive Query Execution (AQE)**
watches the real amount of shuffled data at runtime and can automatically
merge those partitions down to a sensible number - no manual tuning of
`spark.sql.shuffle.partitions` required. We turn it off then on here only to
see the difference; it ships enabled by default.

In [ ]:
import pyspark.sql.functions as sf

spark.conf.set("spark.sql.adaptive.enabled", "false")

with profiler.profile("4-aqe-off"):
    ratings = (spark.read.option("header", "true").schema(ratings_schema)
               .csv(dataFolder + "ratings1.csv.gz").repartition(target_partitions))
    ratings.groupBy("movieId").agg(sf.avg("rating"), sf.count("*")).collect()


26/10/01 09:22:31 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /data/lab02/ratings*.csv.gz.
java.io.FileNotFoundException: File /data/lab02/ratings*.csv.gz does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.

[4-aqe-off] wall time: 49.58s (3 stages, 214 tasks) - see profiler.report() / profiler.compare(...)


`spark.sql.shuffle.partitions` is still at its default (200) here. Check
`profiler.report()`'s `num_tasks` for this run against how many cores your
cluster actually has - and look at the Stages tab: most of those 200 tasks
finish almost instantly because there's so little data in them.

In [15]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

with profiler.profile("4-aqe-on"):
    ratings = (spark.read.option("header", "true").schema(ratings_schema)
               .csv(dataFolder + "ratings1.csv.gz").repartition(target_partitions))
    ratings.groupBy("movieId").agg(sf.avg("rating"), sf.count("*")).collect()


[4-aqe-on] wall time: 44.59s (6 stages, 30 tasks) - see profiler.report() / profiler.compare(...)


In [ ]:
profiler.compare("4-aqe-off", "4-aqe-on")


AQE doesn't reduce *how much* data gets shuffled -
`shuffle_read_bytes`/`shuffle_write_bytes` should be about the same in both
runs - it reduces *how many tasks* that data is split into once Spark can
see the real post-shuffle sizes. That's what `num_tasks` and
`executor_run_time_ms` capture here.

The same mechanism (`spark.sql.adaptive.skewJoin.enabled`, also on by
default) can split an abnormally large shuffle partition instead of merging
small ones - useful when a join or group-by key is skewed, e.g. a handful of
movies with far more ratings than the rest. MovieLens isn't skewed enough
for that half to show up convincingly here, but it's the same knob.

## Example 5 - Python UDFs vs built-in functions

Every example so far uses only built-in Spark functions, which run natively
inside the JVM with Catalyst's optimizations (predicate pushdown, code
generation) intact. A Python UDF breaks that: every row has to be
serialized out of the JVM, handed to a separate Python worker process, and
serialized back. `spark_profiler.py` and the Spark UI still see the
*consequence* of this on the JVM side (`executor_run_time_ms`,
`executor_cpu_time_ms`), but neither can tell you what the Python code
itself is spending its time or memory on. For that, PySpark's own
**Unified UDF Profiler** (`spark.profile`, Spark 4.0+) instruments the
Python worker directly.

In [ ]:
with profiler.profile("5-builtin-bucket"):
    ratings = (spark.read.option("header", "true").schema(ratings_schema)
               .csv(dataFolder + "ratings1.csv.gz").repartition(target_partitions))
    bucketed = ratings.withColumn(
        "bucket",
        sf.when(sf.col("rating") >= 4.5, "excellent")
          .when(sf.col("rating") >= 3.5, "good")
          .when(sf.col("rating") >= 2.5, "average")
          .otherwise("poor"))
    bucketed.groupBy("bucket").count().collect()


In [ ]:
from pyspark.sql.types import StringType

def rating_bucket(rating):
    if rating >= 4.5:
        return "excellent"
    elif rating >= 3.5:
        return "good"
    elif rating >= 2.5:
        return "average"
    return "poor"

rating_bucket_udf = sf.udf(rating_bucket, StringType())

spark.conf.set("spark.sql.pyspark.udf.profiler", "perf")   # "memory" needs `pip install memory_profiler`

with profiler.profile("5-python-udf-bucket"):
    ratings = (spark.read.option("header", "true").schema(ratings_schema)
               .csv(dataFolder + "ratings1.csv.gz").repartition(target_partitions))
    bucketed = ratings.withColumn("bucket", rating_bucket_udf(sf.col("rating")))
    bucketed.groupBy("bucket").count().collect()


In [ ]:
profiler.compare("5-builtin-bucket", "5-python-udf-bucket")


`executor_run_time_ms` and `executor_cpu_time_ms` jump sharply for the UDF
version, even though both do the exact same bucketing over the same 20M
rows - that's the Python round-trip cost `spark_profiler.py` sees from the
JVM side. It can't say anything about what happens *inside*
`rating_bucket` itself, though.

In [ ]:
spark.profile.show()   # the Unified UDF Profiler's own view of the Python side


This is a cProfile-style breakdown (`ncalls`, `tottime`, `cumtime`) of the
Python function itself, per UDF - completely independent of anything
`spark_profiler.py` or the Spark UI measures. The two tools are answering
different questions: one profiles the distributed job around the UDF, the
other profiles the Python code inside it. Swap `"perf"` for `"memory"`
above (after `pip install memory_profiler`) to see line-by-line memory use
instead of call counts/time.

In [ ]:
spark.conf.unset("spark.sql.pyspark.udf.profiler")
spark.profile.clear()


## Summary

In [ ]:
profiler.report()


## Beyond today's examples

* **Storage format** - converting `ratings1.csv.gz` to Parquet once
  (`df.write.mode("overwrite").parquet(...)`) removes the gzip-splittability
  problem at the source and speeds up every subsequent read.
* **Filter pushdown** - a filter applied late in a pipeline (after a join or
  a window function) can let far more data than necessary reach an
  expensive shuffle stage, even though the end result would have been the
  same had it been applied earlier. The exercises notebook has a concrete,
  measurable case of this.
* **Bucketing** - if two tables get joined on the same key over and over
  across many queries, writing them once with `.bucketBy(...)` avoids
  paying the join's shuffle cost every single time.
* **Caching** - any DataFrame reused by more than one action is worth
  `.cache()`-ing once, and `.unpersist()`-ing once you're done with it.
* **Scaling** - try re-running this notebook against a cluster with a
  different number of workers
  (`docker compose up -d --scale spark-worker=N`) and see which examples
  actually benefit from more workers, and which don't.

In [ ]:
spark.stop()
